In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import os


In [3]:
DATA_DIR = '/kaggle/input/sliced/sliced_images_axial'  
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

In [6]:
from torch.utils.data import random_split


In [7]:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_size = len(dataset)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)
test_size = total_size - train_size - val_size 

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)


In [9]:
import json
class_to_idx = dataset.class_to_idx
with open('classes.json', 'w') as f:
    json.dump(class_to_idx, f)


In [10]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.densenet121(pretrained=True)


for param in model.parameters():
    param.requires_grad = False

num_classes = len(class_to_idx)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-4)


In [20]:
import time

total_epochs = 20 
start_epoch = 0   

for epoch in range(start_epoch, total_epochs):
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels).item()
        train_total += labels.size(0)

    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total * 100

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)

    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total * 100

    epoch_end = time.time()
    epoch_duration = epoch_end - epoch_start

   
    m, s = divmod(epoch_duration, 60)
    h, m = divmod(m, 60)

    print(f"Epoch {epoch+1}/{total_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"Time: {int(h)}h {int(m)}m {int(s)}s")


Epoch 1/20 | Train Loss: 1.1124 | Train Acc: 56.36% | Val Loss: 1.1002 | Val Acc: 56.48% | Time: 0h 4m 47s
Epoch 2/20 | Train Loss: 1.0972 | Train Acc: 56.62% | Val Loss: 1.0881 | Val Acc: 56.45% | Time: 0h 5m 4s
Epoch 3/20 | Train Loss: 1.0849 | Train Acc: 56.92% | Val Loss: 1.0777 | Val Acc: 56.98% | Time: 0h 4m 42s
Epoch 4/20 | Train Loss: 1.0761 | Train Acc: 57.36% | Val Loss: 1.0680 | Val Acc: 57.12% | Time: 0h 4m 38s
Epoch 5/20 | Train Loss: 1.0676 | Train Acc: 57.58% | Val Loss: 1.0613 | Val Acc: 57.50% | Time: 0h 4m 35s
Epoch 6/20 | Train Loss: 1.0598 | Train Acc: 57.65% | Val Loss: 1.0575 | Val Acc: 57.66% | Time: 0h 4m 33s
Epoch 7/20 | Train Loss: 1.0535 | Train Acc: 57.94% | Val Loss: 1.0522 | Val Acc: 58.03% | Time: 0h 4m 34s
Epoch 8/20 | Train Loss: 1.0471 | Train Acc: 58.19% | Val Loss: 1.0497 | Val Acc: 57.49% | Time: 0h 4m 36s
Epoch 9/20 | Train Loss: 1.0420 | Train Acc: 58.20% | Val Loss: 1.0443 | Val Acc: 58.04% | Time: 0h 4m 38s
Epoch 10/20 | Train Loss: 1.0404 | Tra